# 04 — AKS kube-audit hunt

Pulls kube-audit events for the **Defender for Containers** scenarios
(S-K8S-01..10): privileged pods, hostPath mounts, exec into pods, cluster-admin
bindings.


In [ ]:
"""Shared setup — run me first.
Connects to Log Analytics via DefaultAzureCredential (uses az login, env vars, or managed identity).
"""
import os
import pandas as pd
from datetime import timedelta
from azure.identity import DefaultAzureCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus

WORKSPACE_ID = os.environ.get("LAW_WORKSPACE_ID")  # GUID of the Log Analytics workspace
assert WORKSPACE_ID, "Set LAW_WORKSPACE_ID env var (workspace customerId GUID, not full ARM id)"

client = LogsQueryClient(DefaultAzureCredential())

def kql(query: str, hours: int = 24) -> pd.DataFrame:
    """Run a KQL query against the workspace and return the first table as a DataFrame."""
    resp = client.query_workspace(WORKSPACE_ID, query, timespan=timedelta(hours=hours))
    if resp.status == LogsQueryStatus.PARTIAL:
        print("WARNING: partial result -", resp.partial_error)
        tables = resp.partial_data
    elif resp.status == LogsQueryStatus.SUCCESS:
        tables = resp.tables
    else:
        raise RuntimeError(f"Query failed: {resp}")
    if not tables:
        return pd.DataFrame()
    t = tables[0]
    return pd.DataFrame(t.rows, columns=t.columns)

pd.set_option("display.max_colwidth", 120)
print("Workspace:", WORKSPACE_ID)


## Privileged pods created

In [ ]:
df = kql("""
AzureDiagnostics
| where Category == "kube-audit"
| where TimeGenerated > ago(24h)
| extend log = parse_json(log_s)
| where log.verb == "create" and log.objectRef.resource == "pods"
| extend Pod = tostring(log.objectRef.name),
         Namespace = tostring(log.objectRef.namespace),
         User = tostring(log.user.username),
         RequestObj = tostring(log.requestObject)
| where RequestObj has '"privileged":true'
| project TimeGenerated, ResourceId, Pod, Namespace, User
""")
df


## Pods with sensitive hostPath mounts

In [ ]:
df = kql("""
AzureDiagnostics
| where Category == "kube-audit"
| where TimeGenerated > ago(24h)
| extend log = parse_json(log_s)
| where log.verb == "create" and log.objectRef.resource == "pods"
| extend HostPaths = extract_all(@'"hostPath":\s*\{[^}]*"path":\s*"([^"]+)"', tostring(log.requestObject))
| mv-expand HostPaths to typeof(string)
| where HostPaths in ("/", "/etc", "/var/run/docker.sock", "/proc")
| project TimeGenerated, Pod=tostring(log.objectRef.name),
          Namespace=tostring(log.objectRef.namespace), HostPaths,
          User=tostring(log.user.username)
""")
df


## kubectl exec activity

In [ ]:
df = kql("""
AzureDiagnostics
| where Category == "kube-audit"
| where TimeGenerated > ago(24h)
| where log_s has '"verb":"create"' and log_s has 'pods/exec'
| extend log = parse_json(log_s)
| project TimeGenerated, Pod=tostring(log.objectRef.name),
          Namespace=tostring(log.objectRef.namespace),
          User=tostring(log.user.username),
          SourceIP=tostring(log.sourceIPs[0])
""")
df


## Privileged role assignments (cluster-admin)

In [ ]:
df = kql("""
AzureDiagnostics
| where Category == "kube-audit"
| where TimeGenerated > ago(24h)
| extend log = parse_json(log_s)
| where log.verb in ("create","update","patch") and log.objectRef.resource in ("clusterrolebindings","rolebindings")
| extend Req = tostring(log.requestObject)
| where Req has "cluster-admin"
| project TimeGenerated, Name=tostring(log.objectRef.name),
          User=tostring(log.user.username), Req
""")
df
